# Xarray with browser-backed Icechunk I/O

`ipygis` is a bridge to GIS libraries running in the browser. There are a couple of things to know to have it working:

- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.
- when using the `@earthmover/icechunk` WASM library, the server must send COOP/COEP headers so that `SharedArrayBuffer` is supported in the browser.

In [ ]:
import numpy as np
from ipygis.icechunk import Repository, jupyter_storage
from ipygis.xarray import open_zarr_async, mosaic_async
from ipygis.zarr import asynchronous as zarr

In [ ]:
storage = jupyter_storage("examples/hydrosheds.icechunk")
repository = await Repository.open_async(
    storage,
    # backend="@earthmover/icechunk",  # "icechunk-js" is the default
    proxy_url="https://my-proxy.david-brochart.workers.dev/",
    virtual_chunk_prefixes=["https://data.hydrosheds.org/file/hydrosheds-v2/ACC/1s/"],
)
session = await repository.readonly_session_async("main")
session.snapshot_id

In [ ]:
ds = await open_zarr_async(session.store)
ds

In [ ]:
# Load only the small arrays describing the geographic origins.
origins = await ds[["tile_x", "tile_y"]].load_async()
raster = ds["0"]
attrs = raster.attrs
dx, dy, _ = attrs["model_pixel_scale"]
if (attrs.get("geographic_type") != 4326 or attrs.get("raster_type") != 1
        or dx <= 0 or dy <= 0 or "model_transformation" in attrs):
    raise ValueError("Expected unrotated WGS84 PixelIsArea tiles")

sources = []
for tile in range(ds.sizes["tile"]):
    west = origins.tile_x.values[tile]
    north = origins.tile_y.values[tile]
    source = raster.isel(tile=tile, drop=True).rename(x="longitude", y="latitude")
    source = source.assign_coords(
        longitude=west + (np.arange(source.sizes["longitude"]) + 0.5) * dx,
        latitude=north - (np.arange(source.sizes["latitude"]) + 0.5) * dy,
    )
    sources.append(source)

mosaic = await mosaic_async(sources, x="longitude", y="latitude", crs="EPSG:4326")
mosaic

In [ ]:
# Choose a small geographic region inside an available tile.
west = float(origins.tile_x.values[0])
north = float(origins.tile_y.values[0])
region = await mosaic.sel(
    longitude=slice(west + 0.01, west + 0.02),
    latitude=slice(north - 0.01, north - 0.02),
).load_async()
region

In [ ]:
point = await mosaic.sel(
    longitude=west + 0.015, latitude=north - 0.015, method="nearest",
).load_async()
point

In [ ]:
group = await zarr.open_group(session.store, mode="r")
array = await group.getitem("0")
array.shape, array.dtype, array.chunks

In [ ]:
result = await array.getitem((10, 100, 200))
result

In [ ]:
# Close after finishing all array reads.
await session.aclose()
await repository.aclose()